In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import sys
sys.path.append("..")

import gc
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1

from utils.load_data import load_data_hf

In [2]:
base_dir = '/groups/chichengz/tnn/datasets/'

# model dirs (per-quantization)
llm_dir_llama_3b_fp16 = base_dir + "Llama3.2-3B-Instruct"
llm_dir_llama_3b_gptq = base_dir + "Llama3.2-3B-Instruct-GPTQ"
llm_dir_qwen_3b_fp16  = base_dir + "Qwen2.5-3B-Instruct"
llm_dir_qwen_3b_gptq  = base_dir + "Qwen2.5-3B-Instruct-GPTQ-Int4"
llm_dir_qwen_7b_fp16  = base_dir + "Qwen2.5-7B-Instruct"
llm_dir_qwen_7b_gptq  = base_dir + "Qwen2.5-7B-Instruct-GPTQ-Int4"

# dataset
ds_split = "test"
ds_dir = base_dir + "/prm800k/math_splits"

In [3]:
# quantization configs to benchmark
# GPTQ requires a pre-quantized model directory (point model_dir to it)
quant_configs = [
    {
        "name":         "llama-3b fp16",
        "model_dir":    llm_dir_llama_3b_fp16,
        "quantization": None,
        "load_format":  "auto",
        "dtype":        "float16",
    },
    {
        "name":         "llama-3b gptq",
        "model_dir":    llm_dir_llama_3b_gptq,
        "quantization": "gptq",
        "load_format":  "auto",
        "dtype":        "auto",
    },
    {
        "name":         "qwen-3b fp16",
        "model_dir":    llm_dir_qwen_3b_fp16,
        "quantization": None,
        "load_format":  "auto",
        "dtype":        "float16",
    },
    {
        "name":         "qwen-3b gptq-int4",
        "model_dir":    llm_dir_qwen_3b_gptq,
        "quantization": "gptq",
        "load_format":  "auto",
        "dtype":        "auto",
    },
    {
        "name":         "qwen-7b fp16",
        "model_dir":    llm_dir_qwen_7b_fp16,
        "quantization": None,
        "load_format":  "auto",
        "dtype":        "float16",
    },
    {
        "name":         "qwen-7b gptq-int4",
        "model_dir":    llm_dir_qwen_7b_gptq,
        "quantization": "gptq",
        "load_format":  "auto",
        "dtype":        "auto",
    },
]

In [ ]:
# general params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048

config.n = 256
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

num_trials = 2
level = 4

llm_gpu_memory_utilization = 0.5

In [5]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)

num_questions = len(dataset)
num_questions = min(num_questions, 5)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 5


### Benchmark `best_of_n_v1` across quantization levels
Each config is loaded, timed, then unloaded before the next to avoid OOM.

In [6]:
def get_gpu_memory_used(device=0):
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


results_summary = []

for qcfg in quant_configs:
    print(f"\n=== {qcfg['name']} ===")

    llm_vllm = LLM(
        model=qcfg["model_dir"],
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype=qcfg["dtype"],
        quantization=qcfg["quantization"],
        load_format=qcfg["load_format"],
        seed=config.seed,
    )

    gpu_mem_gb = get_gpu_memory_used()
    print(f"  GPU memory used after load: {gpu_mem_gb:.2f} GB")

    trial_times = []
    for trial_idx in range(num_trials):
        start_time = time.time()
        bon_search_v1.best_of_n_v1(batch_of_questions, config, llm_vllm, trial_idx)
        elapsed = time.time() - start_time
        trial_times.append(elapsed)
        print(f"  trial {trial_idx}: {elapsed / num_questions:.4f}s/question  {elapsed:.2f}s total")

    avg_time = sum(trial_times) / len(trial_times)
    results_summary.append((qcfg["name"], gpu_mem_gb, avg_time, avg_time / num_questions))

    del llm_vllm
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary ===")
print(f"{'quantization':<25} {'gpu mem (GB)':>12} {'avg s/trial':>12} {'avg s/question':>15}")
print("-" * 67)
for name, mem, avg_trial, avg_q in results_summary:
    print(f"{name:<25} {mem:>12.2f} {avg_trial:>12.2f} {avg_q:>15.4f}")


=== llama-3b fp16 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.50s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.48s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.63s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used after load: 16.91 GB
  trial 0: 11.7248s/question  58.62s total
  trial 1: 12.2862s/question  61.43s total


[rank0]:[W514 16:10:45.782039993 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== llama-3b gptq ===


...


=== qwen-7b gptq-int4 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.99it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.95it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.75it/s]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used after load: 16.51 GB
  trial 0: 19.5365s/question  97.68s total
  trial 1: 20.9526s/question  104.76s total


[rank0]:[W514 16:31:17.971684100 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Summary ===
quantization              gpu mem (GB)  avg s/trial  avg s/question
-------------------------------------------------------------------
llama-3b fp16                    16.91        60.03         12.0055
llama-3b gptq                    17.01        72.36         14.4728
qwen-3b fp16                     16.99        91.39         18.2788
qwen-3b gptq-int4                16.93       104.48         20.8966
qwen-7b fp16                     16.50       164.53         32.9070
qwen-7b gptq-int4                16.51       101.22         20.2446
